In [ ]:
# Muon Panel 3 — Trigger Analysis

Analysis of Xenoscope muon-panel data recorded with Muon Panel 3 as the trigger.

In [ ]:
## Packages

In [ ]:
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import uproot

In [ ]:
## Pulse extraction and processing

In [ ]:
# Folder containing the ROOT files.
# Replace this path with the location of the Xenoscope ROOT data.
DATA_FOLDER = Path("/path/to/Xenoscope/MuonPanelOnly/20260709_131758")

CHANNELS = ["wf0", "wf1", "wf2"]

OUTPUT_FILE = Path("outputfile_muon3.parquet")

# Find all ROOT files in the folder
root_files = sorted(DATA_FOLDER.glob("*.root"))

# Use only the first 15 files
root_files = root_files[:15]

# Initialize result storage
results = {
    channel: {
        "baseline": [],
        "area": [],
        "amplitude": [],
        "nsamples": [],
        "integration_width": [],
    }
    for channel in CHANNELS
}

# Loop over all ROOT files
for filename in root_files:
    print(f"Processing {filename}")

    data = uproot.open(filename)["dig_0"].arrays()

    # Loop over all events
    for event in range(len(data["wf0"])):

        # Loop over all waveform channels
        for channel in CHANNELS:
            waveform = np.asarray(data[channel][event])

            # 1. Calculate baseline from the first 50 samples
            baseline = np.mean(waveform[:50])

            # 2. Define threshold for negative-going pulses
            threshold = baseline - 10

            # 3. Find samples below threshold
            indices = np.where(waveform < threshold)[0]

            if len(indices) > 0:
                # Define integration window
                start = max(0, indices[0] - 3)
                stop = min(len(waveform), indices[-1] + 4)

                # Calculate pulse quantities
                area = np.sum(baseline - waveform[start:stop])
                amplitude = baseline - np.min(waveform[start:stop])

                # Additional quantities
                nsamples = len(indices)
                integration_width = stop - start

            else:
                area = 0
                amplitude = 0
                nsamples = 0
                integration_width = 0

            # Store quantities
            results[channel]["baseline"].append(baseline)
            results[channel]["area"].append(area)
            results[channel]["amplitude"].append(amplitude)
            results[channel]["nsamples"].append(nsamples)
            results[channel]["integration_width"].append(
                integration_width
            )

# Convert lists to NumPy arrays
for channel in CHANNELS:
    for key in results[channel]:
        results[channel][key] = np.asarray(results[channel][key])

# Convert to an Awkward Array
output = ak.Array(results)

# Save to Parquet
ak.to_parquet(output, OUTPUT_FILE)

print(f"Results written to {OUTPUT_FILE}")

In [ ]:
## Load processed data

In [ ]:
data = ak.from_parquet("outputfile_muon3.parquet")

In [ ]:
## A. Baseline distribution

In [ ]:
baselines0 = np.asarray(data["wf0"]["baseline"])
baselines1 = np.asarray(data["wf1"]["baseline"])
baselines2 = np.asarray(data["wf2"]["baseline"])

# Determine a common histogram range
xmin = min(
    baselines0.min(),
    baselines1.min(),
    baselines2.min(),
)
xmax = max(
    baselines0.max(),
    baselines1.max(),
    baselines2.max(),
)

bins = np.linspace(xmin, xmax, 161)

plt.figure(figsize=(8, 5))

plt.hist(baselines0, bins=bins, alpha=0.5, label="wf0")
plt.hist(baselines1, bins=bins, alpha=0.5, label="wf1")
plt.hist(baselines2, bins=bins, alpha=0.5, label="wf2")

plt.xlim(14999, 15005)

plt.xlabel("Baseline (ADC)")
plt.ylabel("Number of events")
plt.title("Baseline distribution")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
## B. Integrated signal area

### B. i. Linear scale

In [ ]:
areas0 = np.asarray(data["wf0"]["area"])
areas1 = np.asarray(data["wf1"]["area"])
areas2 = np.asarray(data["wf2"]["area"])

# Number of events with a detected pulse
trigger0 = np.sum(areas0 > 0)
trigger1 = np.sum(areas1 > 0)
trigger2 = np.sum(areas2 > 0)

bins = np.linspace(-25, 1000, 161)

plt.figure(figsize=(8, 5))

plt.hist(
    areas0,
    bins=bins,
    alpha=0.5,
    label=f"wf0 ({trigger0} triggers)",
)
plt.hist(
    areas1,
    bins=bins,
    alpha=0.5,
    label=f"wf1 ({trigger1} triggers)",
)
plt.hist(
    areas2,
    bins=bins,
    alpha=0.5,
    label=f"wf2 ({trigger2} triggers)",
)

plt.xlim(-25, 1000)
plt.ylim(0, 10000)

plt.xlabel("Signal area (ADC × samples)")
plt.ylabel("Number of events")
plt.title("Integrated signal area")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
### B. ii. Logarithmic scale

In [ ]:
bins = np.linspace(-25, 750, 161)

plt.figure(figsize=(8, 5))

plt.hist(
    areas0,
    bins=bins,
    alpha=0.5,
    label=f"wf0 ({trigger0} triggers)",
)
plt.hist(
    areas1,
    bins=bins,
    alpha=0.5,
    label=f"wf1 ({trigger1} triggers)",
)
plt.hist(
    areas2,
    bins=bins,
    alpha=0.5,
    label=f"wf2 ({trigger2} triggers)",
)

plt.yscale("log")
plt.xlim(-100, 500)

plt.xlabel("Signal area (ADC × samples)")
plt.ylabel("Number of events")
plt.title("Integrated signal area")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
## C. Pulse amplitude

### C. i. Linear scale

In [ ]:
amplitudes0 = np.asarray(data["wf0"]["amplitude"])
amplitudes1 = np.asarray(data["wf1"]["amplitude"])
amplitudes2 = np.asarray(data["wf2"]["amplitude"])

# Number of events with a detected pulse
trigger0 = np.sum(amplitudes0 > 0)
trigger1 = np.sum(amplitudes1 > 0)
trigger2 = np.sum(amplitudes2 > 0)

bins = np.linspace(0, 300, 101)

plt.figure(figsize=(8, 5))

plt.hist(
    amplitudes0,
    bins=bins,
    alpha=0.5,
    label=f"wf0 ({trigger0} triggers)",
)
plt.hist(
    amplitudes1,
    bins=bins,
    alpha=0.5,
    label=f"wf1 ({trigger1} triggers)",
)
plt.hist(
    amplitudes2,
    bins=bins,
    alpha=0.5,
    label=f"wf2 ({trigger2} triggers)",
)

plt.xlim(0, 400)
plt.ylim(0, 20000)

plt.xlabel("Pulse amplitude (ADC)")
plt.ylabel("Number of events")
plt.title("Pulse amplitude distribution")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
### C. ii. Logarithmic scale

In [ ]:
bins = np.linspace(5, 3000, 301)

plt.figure(figsize=(8, 5))

plt.hist(
    amplitudes0,
    bins=bins,
    alpha=0.5,
    label=f"wf0 ({trigger0} triggers)",
)
plt.hist(
    amplitudes1,
    bins=bins,
    alpha=0.5,
    label=f"wf1 ({trigger1} triggers)",
)
plt.hist(
    amplitudes2,
    bins=bins,
    alpha=0.5,
    label=f"wf2 ({trigger2} triggers)",
)

plt.yscale("log")
plt.xlim(5, 1000)

plt.xlabel("Pulse amplitude (ADC)")
plt.ylabel("Number of events")
plt.title("Pulse amplitude distribution")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
## D. Pulse-area correlation between wf0 and wf1

In [ ]:
area0 = ak.to_numpy(data["wf0"]["area"])
area1 = ak.to_numpy(data["wf1"]["area"])

# Keep only events with a detected pulse in both channels
mask = (area0 > 0) & (area1 > 0)

area0 = area0[mask]
area1 = area1[mask]

plt.figure(figsize=(7, 6))

plt.hist2d(
    area0,
    area1,
    bins=2000,
    range=[
        [0, np.percentile(area0, 99.5)],
        [0, np.percentile(area1, 99.5)],
    ],
    cmap="viridis",
    cmin=1,
)

plt.xlim(0, 400)
plt.ylim(0, 400)

plt.colorbar(label="Counts")
plt.xlabel("Pulse area wf0 (ADC × samples)")
plt.ylabel("Pulse area wf1 (ADC × samples)")
plt.title("wf0 pulse area vs. wf1 pulse area")

plt.tight_layout()
plt.show()

In [ ]:
## E. Integration width vs pulse amplitude

In [ ]:
plt.figure(figsize=(8, 5))

for channel in CHANNELS:

    amplitudes = np.asarray(
        ak.to_numpy(
            ak.flatten(
                data[channel]["amplitude"],
                axis=None,
            )
        ),
        dtype=float,
    )

    widths = np.asarray(
        ak.to_numpy(
            ak.flatten(
                data[channel]["integration_width"],
                axis=None,
            )
        ),
        dtype=float,
    )

    # Keep only valid events with a detected pulse
    mask = (
        np.isfinite(amplitudes)
        & np.isfinite(widths)
        & (amplitudes > 0)
        & (widths > 0)
    )

    amplitudes = amplitudes[mask]
    widths = widths[mask]

    plt.scatter(
        amplitudes,
        widths,
        s=8,
        alpha=0.5,
        label=channel,
    )

plt.xlim(0, 1000)
plt.ylim(0, 100)

plt.xlabel("Pulse amplitude (ADC)")
plt.ylabel("Integration width (samples)")
plt.title("Pulse width vs. pulse amplitude")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()